# RAG Application with LlamaIndex + OpenAI

Ask questions over your own PDFs using Retrieval-Augmented Generation.

**Before you run:**
1. Add your OpenAI key to Colab Secrets (🔑 icon in the left sidebar) with the name `OPENAI_API_KEY` and enable *Notebook access*. (If you skip this, you'll be prompted to paste it.)
2. Have your PDFs ready to upload (e.g. the Transformers and YOLO papers).
3. Run the cells top to bottom (`Runtime → Run all` works once your PDFs are uploaded).

## 0. Install dependencies

In [ ]:
%pip install -q llama-index llama-index-embeddings-openai llama-index-llms-openai pypdf python-dotenv

## 1. Environment setup & API key

In [ ]:
import os
import os.path
from getpass import getpass

# Try Colab Secrets first, then an existing env var, then ask interactively.
api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
except Exception:
    pass

api_key = api_key or os.getenv("OPENAI_API_KEY") or getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key
print("OPENAI_API_KEY is set ✅")

## 2. Document loading & vector indexing

First upload your PDFs into a `data/` folder.

In [ ]:
from google.colab import files
import shutil

os.makedirs("data", exist_ok=True)

uploaded = files.upload()  # pick one or more PDF files
for name in uploaded:
    shutil.move(name, os.path.join("data", name))

print("Files in data/:", os.listdir("data"))

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

# Load all files inside the data directory
documents = SimpleDirectoryReader("data").load_data()
print(f"Loaded {len(documents)} document pages/sections")

# Build the vector store index (embeds every chunk with OpenAI embeddings)
index = VectorStoreIndex.from_documents(documents, show_progress=True)

## 3. Basic query engine

> The example questions assume you uploaded the Transformers and YOLO papers. Change them to match your own documents.

In [ ]:
query_engine = index.as_query_engine()

response_transformers = query_engine.query("What is Transformers")
print(response_transformers)

print("-" * 60)

response_yolo = query_engine.query("What is YOLO")
print(response_yolo)

## 4. Custom retriever & similarity post-processing

Retrieves the top 4 chunks (`similarity_top_k=4`) and drops any chunk scoring below the similarity cutoff.

In [ ]:
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SimilarityPostprocessor

SIMILARITY_CUTOFF = 0.80  # lower this (e.g. 0.70) if you get empty responses

# Retrieve the top 4 matching nodes
retriever = VectorIndexRetriever(index=index, similarity_top_k=4)

# Filter out chunks below the cutoff
postprocessor = SimilarityPostprocessor(similarity_cutoff=SIMILARITY_CUTOFF)

custom_query_engine = RetrieverQueryEngine(
    retriever=retriever,
    node_postprocessors=[postprocessor],
)

response = custom_query_engine.query("What is attention is all you need")
print(response)

### Tuning the cutoff: inspect similarity scores

OpenAI embedding scores often fall between 0.70 and 0.85, so a cutoff of 0.80 can be strict. This cell shows the raw scores of the retrieved chunks so you can pick a sensible threshold.

In [ ]:
question = "What is attention is all you need"

for i, node in enumerate(retriever.retrieve(question), start=1):
    preview = node.node.get_content()[:120].replace("\n", " ")
    print(f"{i}. score={node.score:.3f} | {preview}...")

## 5. Persist the index to disk & load it back

Saving the index avoids re-embedding your documents every time. 

> **Note:** Colab's local disk is wiped when the runtime resets. To keep the index between sessions, uncomment the Google Drive lines below so it's stored in your Drive.

In [ ]:
from llama_index.core import StorageContext, load_index_from_storage

# --- Optional: persist to Google Drive so it survives runtime resets ---
# from google.colab import drive
# drive.mount("/content/drive")
# PERSIST_DIR = "/content/drive/MyDrive/rag_storage"

PERSIST_DIR = "./storage"

if not os.path.exists(PERSIST_DIR):
    # First run: save the index we already built in step 2
    index.storage_context.persist(persist_dir=PERSIST_DIR)
    print(f"Index saved to {PERSIST_DIR}")
else:
    print(f"Found an existing index at {PERSIST_DIR}")

# Load it back from disk (no re-embedding needed)
storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
loaded_index = load_index_from_storage(storage_context)

persistent_query_engine = loaded_index.as_query_engine()
response = persistent_query_engine.query("What is attention is all you need")
print(response)

## 6. Ask your own questions

In [ ]:
my_question = "Summarize the main contribution of the documents"  # <- edit me

response = persistent_query_engine.query(my_question)
print(response)

print("\nSources:")
for n in response.source_nodes:
    print(f"- {n.node.metadata.get('file_name', 'unknown')} "
          f"(page {n.node.metadata.get('page_label', '?')}, score={n.score:.3f})")